# RISK_DATA_V6 — ABS 2% Label + Expanded Features + Ablation

- **Label:** `big_move_next = 1` if `|logret_next| >= 0.02` (2%).
- **Expanded features:** ATR/True Range, RSI, momentum, gap %, rolling corr/beta vs index, volatility regime percentile.
- **Pipeline:** streaming scaler (train-only) + tf.data generator (RAM-safe).
- **Models:** Baseline LSTM vs MSF-CAL inspired.


In [2]:
# =========================
# Cell 1 — V6 Stock-only (ABS 2%) with sanitation + clipping (NO index)
# =========================
!pip -q install yfinance tqdm

import os
import numpy as np
import pandas as pd
import yfinance as yf
from tqdm import tqdm

START = "2010-01-01"      # keep stable; after it works you can move earlier
END   = "2025-01-01"
ABS_THR = 0.02            # 2% absolute next-day move label
MIN_LEN = 500             # keep enough after indicators

TICKERS_CSV = "tickers_500.txt"
SEED_TICKERS = [
    "RELIANCE.NS","TCS.NS","HDFCBANK.NS","INFY.NS","ICICIBANK.NS","ITC.NS",
    "KOTAKBANK.NS","LT.NS","SBIN.NS","AXISBANK.NS","BHARTIARTL.NS","ASIANPAINT.NS"
]

def load_tickers(csv_path=TICKERS_CSV):
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        col = "ticker" if "ticker" in df.columns else df.columns[0]
        tickers = df[col].dropna().astype(str).str.strip().tolist()
        print(f"Loaded {len(tickers)} tickers from {csv_path} ({col} column)")
        return tickers
    print(f"Could not load {csv_path}. Using SEED_TICKERS instead.")
    return SEED_TICKERS

TICKERS = load_tickers()

def batch_download(tickers, start=START, end=END):
    # One call => far less Yahoo throttling than 100 per-ticker calls
    df = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        progress=False,
        auto_adjust=False,
        group_by="ticker",
        threads=True
    )
    return df

def get_one_ticker_ohlcv(big_df, ticker: str) -> pd.DataFrame:
    if big_df is None or big_df.empty:
        return pd.DataFrame()

    # Extract per-ticker frame from MultiIndex
    if not isinstance(big_df.columns, pd.MultiIndex):
        df = big_df.copy()
    else:
        df = None
        lv0 = big_df.columns.get_level_values(0)
        lv1 = big_df.columns.get_level_values(1)
        if ticker in lv0:
            # (Ticker, Field)
            df = big_df[ticker].copy()
        elif ticker in lv1:
            # (Field, Ticker)
            df = big_df.xs(ticker, axis=1, level=1, drop_level=True).copy()
        else:
            return pd.DataFrame()

    # Normalize column names
    rename = {}
    for c in df.columns:
        low = str(c).lower()
        if low.startswith("open"): rename[c] = "open"
        elif low.startswith("high"): rename[c] = "high"
        elif low.startswith("low"): rename[c] = "low"
        elif low.startswith("close") and "adj" not in low: rename[c] = "close"
        elif "volume" in low: rename[c] = "volume"
    df = df.rename(columns=rename)

    need = ["open","high","low","close"]
    if any(c not in df.columns for c in need):
        return pd.DataFrame()

    if "volume" not in df.columns:
        df["volume"] = np.nan

    df = df[["open","high","low","close","volume"]].copy()
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Require OHLC; volume can be missing
    df = df.dropna(subset=["open","high","low","close"])
    df["volume"] = df["volume"].fillna(0.0)

    # Remove non-positive close (prevents log issues)
    df = df[df["close"] > 0]

    return df

def safe_clip(s: pd.Series, lo: float, hi: float) -> pd.Series:
    return s.clip(lower=lo, upper=hi)

def add_features_and_label(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Core returns
    df["logret"] = np.log(df["close"] / df["close"].shift(1))
    df["ret_1d"] = df["close"].pct_change(1)
    df["ret_5d"] = df["close"].pct_change(5)
    df["mom_10"] = df["close"] / df["close"].shift(10) - 1.0

    # MAs
    df["ma10"]  = df["close"].rolling(10,  min_periods=10).mean()
    df["ma50"]  = df["close"].rolling(50,  min_periods=50).mean()
    df["ma100"] = df["close"].rolling(100, min_periods=100).mean()

    # Volatility
    df["vol_10"] = df["logret"].rolling(10, min_periods=10).std()
    df["vol_20"] = df["logret"].rolling(20, min_periods=20).std()
    df["vol_60"] = df["logret"].rolling(60, min_periods=60).std()

    # Volume z-score (safer): use log1p(volume) to reduce huge outliers, avoid std=0 blowups
    v = np.log1p(df["volume"])
    v_ma  = v.rolling(20, min_periods=20).mean()
    v_std = v.rolling(20, min_periods=20).std()
    df["volume_z"] = (v - v_ma) / (v_std + 1e-6)

    # Trend (safe division)
    df["trend_10_50"] = (df["ma10"] - df["ma50"]) / (df["ma50"].abs() + 1e-6)

    # Gap %
    df["gap_pct"] = (df["open"] / df["close"].shift(1)) - 1.0

    # ATR14
    prev_close = df["close"].shift(1)
    tr = np.maximum(
        df["high"] - df["low"],
        np.maximum((df["high"] - prev_close).abs(), (df["low"] - prev_close).abs())
    )
    df["atr14"] = pd.Series(tr, index=df.index).rolling(14, min_periods=14).mean()

    # RSI14
    delta = df["close"].diff()
    up = delta.clip(lower=0)
    down = (-delta).clip(lower=0)
    roll_up = up.rolling(14, min_periods=14).mean()
    roll_down = down.rolling(14, min_periods=14).mean()
    rs = roll_up / (roll_down + 1e-6)
    df["rsi14"] = 100 - (100 / (1 + rs))

    # Volatility regime (stock-only), but don’t force 252 days; compute with min_periods=60
    W = 252
    def pct_rank_last(x):
        x = np.asarray(x)
        return float(np.mean(x <= x[-1]))
    df["vol_regime_252"] = df["vol_20"].rolling(W, min_periods=60).apply(pct_rank_last, raw=True)
    df["vol_regime_252"] = df["vol_regime_252"].fillna(0.5)

    # Label
    df["logret_next"] = df["logret"].shift(-1)
    df["big_move_next"] = (df["logret_next"].abs() >= ABS_THR).astype(int)

    # --- SANITIZE ---
    df = df.replace([np.inf, -np.inf], np.nan)

    # Drop only essential fields for model + label
    essential = [
        "logret","ret_1d","ret_5d","mom_10",
        "ma10","ma50","ma100",
        "vol_10","vol_20","vol_60",
        "volume_z","trend_10_50","gap_pct","atr14","rsi14","vol_regime_252",
        "big_move_next"
    ]
    df = df.dropna(subset=essential)

    # Clip outliers to prevent gradient blow-ups
    df["volume_z"]      = safe_clip(df["volume_z"], -8, 8)
    df["gap_pct"]       = safe_clip(df["gap_pct"], -0.25, 0.25)
    df["ret_1d"]        = safe_clip(df["ret_1d"], -0.25, 0.25)
    df["ret_5d"]        = safe_clip(df["ret_5d"], -0.50, 0.50)
    df["mom_10"]        = safe_clip(df["mom_10"], -0.80, 0.80)
    df["trend_10_50"]   = safe_clip(df["trend_10_50"], -2.0, 2.0)
    df["rsi14"]         = safe_clip(df["rsi14"], 0.0, 100.0)
    df["vol_regime_252"]= safe_clip(df["vol_regime_252"], 0.0, 1.0)

    return df

# ---- Batch download once ----
big = batch_download(TICKERS, start=START, end=END)
print("Batch download shape:", big.shape)

# ---- Build per-stock frames ----
stock_frames = {}
pos_rates = []

for t in tqdm(TICKERS):
    ohlcv = get_one_ticker_ohlcv(big, t)
    if ohlcv.empty or len(ohlcv) < 300:
        continue

    feat = add_features_and_label(ohlcv)
    if len(feat) < MIN_LEN:
        continue

    stock_frames[t] = feat
    pos_rates.append(float(feat["big_move_next"].mean()))

if len(stock_frames) == 0:
    raise ValueError("No tickers kept. Try re-running cell or setting START='2012-01-01' to reduce download size.")

print(f"\nABS threshold: {ABS_THR:.2%}")
print(f"Built frames for {len(stock_frames)} tickers.")
print(f"Avg pos-rate: {np.mean(pos_rates):.4f} (min={np.min(pos_rates):.4f}, max={np.max(pos_rates):.4f})")

# sanity: check NaN/Inf in one ticker’s features
tt = next(iter(stock_frames))
Xcheck = stock_frames[tt][[
    "logret","ret_1d","ret_5d","mom_10","ma10","ma50","ma100","vol_10","vol_20","vol_60",
    "volume_z","trend_10_50","gap_pct","atr14","rsi14","vol_regime_252"
]].to_numpy()
print("Sanity (one ticker): NaN =", np.isnan(Xcheck).sum(), "Inf =", np.isinf(Xcheck).sum(), "rows =", Xcheck.shape[0])


Loaded 499 tickers from tickers_500.txt (360ONE.NS column)


ERROR:yfinance:
13 Failed downloads:
ERROR:yfinance:['TMPV.NS', 'ATHERENERG.NS', 'AEGISVOPAK.NS', 'AGARWALEYE.NS', 'ONESOURCE.NS', 'THELEELA.NS', 'ITCHOTELS.NS', 'ABLBL.NS', 'ENRIN.NS', 'LTM.NS', 'COHANCE.NS', 'JSWCEMENT.NS', 'HEXT.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2010-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1262284200, endDate = 1735669800")')


Batch download shape: (3704, 2994)


100%|██████████| 499/499 [00:24<00:00, 19.99it/s]


ABS threshold: 2.00%
Built frames for 427 tickers.
Avg pos-rate: 0.2923 (min=0.1119, max=0.5292)
Sanity (one ticker): NaN = 0 Inf = 0 rows = 3601


In [3]:
# =========================
# Cell 2 — Streaming scaler + tf.data datasets (RAM-safe) + NaN/Inf guard
# =========================
import math
import numpy as np
import tensorflow as tf

T = 120

FEATURE_COLS = [
    "logret",
    "ret_1d", "ret_5d", "mom_10",
    "ma10", "ma50", "ma100",
    "vol_10", "vol_20", "vol_60",
    "volume_z",
    "trend_10_50",
    "gap_pct",
    "atr14",
    "rsi14",
    "vol_regime_252",
]
LABEL_COL = "big_move_next"

BATCH_SIZE = 256
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15

F = len(FEATURE_COLS)
print("Features (F):", F)

def iter_windows_for_stock(df, split: str):
    feats = df[FEATURE_COLS].to_numpy(dtype=np.float32, copy=False)
    labels = df[LABEL_COL].to_numpy(dtype=np.int8, copy=False)

    n = len(df)
    if n <= T + 1:
        return

    i_start = T
    i_end = n
    total = i_end - i_start

    tr_end = i_start + int(total * TRAIN_FRAC)
    va_end = tr_end + int(total * VAL_FRAC)

    if split == "train":
        lo, hi = i_start, tr_end
    elif split == "val":
        lo, hi = tr_end, va_end
    elif split == "test":
        lo, hi = va_end, i_end
    else:
        raise ValueError("split must be train/val/test")

    for i in range(lo, hi):
        yield feats[i - T:i], labels[i]

def iter_windows_all_stocks(stock_frames, split: str):
    for _t, df in stock_frames.items():
        yield from iter_windows_for_stock(df, split=split)

class StreamingStandardizer:
    def __init__(self, F):
        self.count = 0
        self.mean = np.zeros(F, dtype=np.float64)
        self.M2 = np.zeros(F, dtype=np.float64)

    def update_batch(self, X_2d):
        X = X_2d.astype(np.float64, copy=False)
        for x in X:
            self.count += 1
            delta = x - self.mean
            self.mean += delta / self.count
            delta2 = x - self.mean
            self.M2 += delta * delta2

    def finalize(self):
        var = self.M2 / max(self.count - 1, 1)
        std = np.sqrt(var)
        # avoid div-by-zero
        std[std == 0] = 1.0
        # avoid NaN std
        std = np.where(np.isfinite(std), std, 1.0)
        mean = np.where(np.isfinite(self.mean), self.mean, 0.0)
        return mean.astype(np.float32), std.astype(np.float32)

print("Computing scaler stats from TRAIN split (streaming)...")
scaler = StreamingStandardizer(F)

chunk = []
CHUNK_WINDOWS = 64
for X, _y in iter_windows_all_stocks(stock_frames, split="train"):
    chunk.append(X)
    if len(chunk) >= CHUNK_WINDOWS:
        scaler.update_batch(np.concatenate(chunk, axis=0))
        chunk = []
if chunk:
    scaler.update_batch(np.concatenate(chunk, axis=0))

mean, std = scaler.finalize()
print("Scaler fitted. count rows:", scaler.count)

def make_tf_dataset(stock_frames, split: str, mean, std, shuffle=False):
    output_signature = (
        tf.TensorSpec(shape=(T, F), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int8),
    )
    def gen():
        for X, y in iter_windows_all_stocks(stock_frames, split=split):
            Xs = (X - mean) / std
            yield Xs, y
    ds = tf.data.Dataset.from_generator(gen, output_signature=output_signature)
    if shuffle:
        ds = ds.shuffle(20000, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_tf_dataset(stock_frames, "train", mean, std, shuffle=True).repeat()
val_ds   = make_tf_dataset(stock_frames, "val",   mean, std, shuffle=False).repeat()
test_ds  = make_tf_dataset(stock_frames, "test",  mean, std, shuffle=False).repeat()

def count_windows(stock_frames, split: str):
    total = 0
    for _t, df in stock_frames.items():
        n = len(df)
        if n <= T + 1:
            continue
        i_start = T
        i_end = n
        tot = i_end - i_start
        tr_end = i_start + int(tot * TRAIN_FRAC)
        va_end = tr_end + int(tot * VAL_FRAC)
        if split == "train":
            total += max(0, tr_end - i_start)
        elif split == "val":
            total += max(0, va_end - tr_end)
        else:
            total += max(0, i_end - va_end)
    return total

train_steps = math.ceil(count_windows(stock_frames, "train") / BATCH_SIZE)
val_steps   = math.ceil(count_windows(stock_frames, "val")   / BATCH_SIZE)
test_steps  = math.ceil(count_windows(stock_frames, "test")  / BATCH_SIZE)

xb, yb = next(iter(train_ds))
xnp = xb.numpy()
print("Steps:", train_steps, val_steps, test_steps)
print("Batch X:", xb.shape, "| Pos rate:", float(tf.reduce_mean(tf.cast(yb, tf.float32)).numpy()))
print("NaNs in batch:", np.isnan(xnp).sum(), "Infs:", np.isinf(xnp).sum())
assert np.isnan(xnp).sum() == 0 and np.isinf(xnp).sum() == 0, "NaN/Inf still present in dataset!"


Features (F): 16
Computing scaler stats from TRAIN split (streaming)...
Scaler fitted. count rows: 102940200
Steps: 3351 718 720
Batch X: (256, 120, 16) | Pos rate: 0.27734375
NaNs in batch: 0 Infs: 0


In [4]:
# =========================
# Cell 3 — Train BOTH models (Ablation) + PR-AUC + class weights + threshold tuning
# (with gradient clipping to avoid instability)
# =========================
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

tf.keras.backend.clear_session()
print("TF version:", tf.__version__)

xb, yb = next(iter(train_ds))
T, F = int(xb.shape[1]), int(xb.shape[2])
input_shape = (T, F)
print("Input shape:", input_shape)

def compute_class_weight_from_train(train_ds, train_steps):
    pos = 0
    tot = 0
    for _x, y in train_ds.take(train_steps):
        y = y.numpy()
        pos += int(y.sum())
        tot += int(len(y))
    neg = tot - pos
    w0 = tot / (2.0 * max(neg, 1))
    w1 = tot / (2.0 * max(pos, 1))
    return {0: w0, 1: w1}, (pos / max(tot, 1))

class_weight, pos_rate = compute_class_weight_from_train(train_ds, train_steps)
print("Train pos rate (approx 1 epoch):", pos_rate)
print("Class weights:", class_weight)

def build_lstm_model(input_shape, l2=1e-4, dropout=0.2):
    inp = layers.Input(shape=input_shape)
    x = layers.LSTM(64, return_sequences=True, kernel_regularizer=regularizers.l2(l2))(inp)
    x = layers.Dropout(dropout)(x)
    x = layers.LSTM(32, kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(l2))(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return models.Model(inp, out, name="baseline_lstm")

class AttentionPooling(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.score = layers.Dense(1)
    def call(self, x):
        scores = self.score(x)
        weights = tf.nn.softmax(scores, 1)
        return tf.reduce_sum(weights * x, 1)

def build_msfcal_model(input_shape, l2=1e-4, dropout=0.25, conv_filters=32, lstm_units=64):
    inp = layers.Input(shape=input_shape)
    x0 = layers.LayerNormalization()(inp)

    branches = []
    for k in (3, 5, 7):
        b = layers.Conv1D(conv_filters, kernel_size=k, padding="same",
                          activation="relu",
                          kernel_regularizer=regularizers.l2(l2))(x0)
        b = layers.Dropout(dropout)(b)
        branches.append(b)

    x = layers.Concatenate()(branches)
    x = layers.Conv1D(conv_filters, kernel_size=1, padding="same",
                      activation="relu",
                      kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True, kernel_regularizer=regularizers.l2(l2))
    )(x)
    x = layers.Dropout(dropout)(x)

    x = AttentionPooling()(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return models.Model(inp, out, name="msfcal")

def compile_model(m):
    opt = tf.keras.optimizers.Adam(1e-3, clipnorm=1.0)  # gradient clipping helps stability
    m.compile(
        optimizer=opt,
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.AUC(name="prauc", curve="PR"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )
    return m

def tune_threshold(model, val_ds, val_steps):
    y_true, y_prob = [], []
    for x, y in val_ds.take(val_steps):
        p = model.predict(x, verbose=0).reshape(-1)
        y_true.append(y.numpy()); y_prob.append(p)
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)

    best = None
    for thr in np.linspace(0.05, 0.95, 19):
        yp = (y_prob >= thr).astype(int)
        tp = ((yp==1)&(y_true==1)).sum()
        fp = ((yp==1)&(y_true==0)).sum()
        fn = ((yp==0)&(y_true==1)).sum()
        precision = tp/(tp+fp+1e-9)
        recall    = tp/(tp+fn+1e-9)
        f1        = 2*precision*recall/(precision+recall+1e-9)
        if best is None or f1 > best["f1"]:
            best = {"thr": float(thr), "precision": float(precision), "recall": float(recall), "f1": float(f1)}
    return best

def eval_with_threshold(model, ds, steps, thr):
    y_true, y_prob = [], []
    for x, y in ds.take(steps):
        p = model.predict(x, verbose=0).reshape(-1)
        y_true.append(y.numpy()); y_prob.append(p)
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)

    yp = (y_prob >= thr).astype(int)
    tp = ((yp==1)&(y_true==1)).sum()
    fp = ((yp==1)&(y_true==0)).sum()
    fn = ((yp==0)&(y_true==1)).sum()
    precision = tp/(tp+fp+1e-9)
    recall    = tp/(tp+fn+1e-9)
    f1        = 2*precision*recall/(precision+recall+1e-9)
    return {"precision_thr": float(precision), "recall_thr": float(recall), "f1_thr": float(f1)}

def train_and_eval(model, tag, epochs=25, save_dir="models_v6_fixed_abs2"):
    os.makedirs(save_dir, exist_ok=True)
    compile_model(model)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_auc", patience=5, mode="max", restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", patience=2, mode="max", factor=0.5, min_lr=1e-5),
    ]

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        steps_per_epoch=train_steps,
        validation_steps=val_steps,
        callbacks=callbacks,
        class_weight=class_weight,
        verbose=1
    )

    test_metrics = model.evaluate(test_ds, steps=test_steps, verbose=0)
    test_dict = dict(zip(model.metrics_names, test_metrics))

    best_thr = tune_threshold(model, val_ds, val_steps)
    thr_test = eval_with_threshold(model, test_ds, test_steps, best_thr["thr"])

    weights_path = os.path.join(save_dir, f"{tag}.weights.h5")
    model.save_weights(weights_path)

    return test_dict, best_thr, thr_test, weights_path

results = []

m1 = build_lstm_model(input_shape)
print("\n" + "="*80 + f"\nTraining baseline_lstm (params={m1.count_params():,})\n" + "="*80)
t1, thr1, thrtest1, w1 = train_and_eval(m1, "baseline_lstm")
results.append({"model":"baseline_lstm","params":m1.count_params(), **t1,
                "best_val_thr":thr1["thr"], "val_f1@thr":thr1["f1"],
                "test_precision@thr":thrtest1["precision_thr"], "test_recall@thr":thrtest1["recall_thr"], "test_f1@thr":thrtest1["f1_thr"],
                "weights":w1})

m2 = build_msfcal_model(input_shape)
print("\n" + "="*80 + f"\nTraining msfcal (params={m2.count_params():,})\n" + "="*80)
t2, thr2, thrtest2, w2 = train_and_eval(m2, "msfcal")
results.append({"model":"msfcal","params":m2.count_params(), **t2,
                "best_val_thr":thr2["thr"], "val_f1@thr":thr2["f1"],
                "test_precision@thr":thrtest2["precision_thr"], "test_recall@thr":thrtest2["recall_thr"], "test_f1@thr":thrtest2["f1_thr"],
                "weights":w2})

df_results = pd.DataFrame(results)
display(df_results)


TF version: 2.19.0
Input shape: (120, 16)
Train pos rate (approx 1 epoch): 0.2980969533768149
Class weights: {0: 0.7123490949433415, 1: 1.6773066424733496}

Training baseline_lstm (params=34,241)
Epoch 1/25
3351/3351 ━━━━━━━━━━━━━━━━━━━━ 400s 115ms/step - auc: 0.6496 - loss: 0.6625 - prauc: 0.4360 - precision: 0.3948 - recall: 0.6010 - val_auc: 0.6480 - val_loss: 0.6716 - val_prauc: 0.4481 - val_precision: 0.3866 - val_recall: 0.6616 - learning_rate: 0.0010
Epoch 2/25
3351/3351 ━━━━━━━━━━━━━━━━━━━━ 387s 116ms/step - auc: 0.6583 - loss: 0.6527 - prauc: 0.4495 - precision: 0.4024 - recall: 0.6017 - val_auc: 0.6497 - val_loss: 0.6549 - val_prauc: 0.4511 - val_precision: 0.4010 - val_recall: 0.6002 - learning_rate: 0.0010
Epoch 3/25
3351/3351 ━━━━━━━━━━━━━━━━━━━━ 390s 116ms/step - auc: 0.6591 - loss: 0.6516 - prauc: 0.4512 - precision: 0.4043 - recall: 0.5923 - val_auc: 0.6502 - val_loss: 0.6670 - val_prauc: 0.4522 - val_precision: 0.3919 - val_recall: 0.6430 - learning_rate: 0.0010
Epoch 

,model,params,loss,compile_metrics,best_val_thr,val_f1@thr,test_precision@thr,test_recall@thr,test_f1@thr,weights
0,baseline_lstm,34241,0.622033,0.659279,0.45,0.495131,0.349605,0.642387,0.452789,models_v6_fixed_abs2/baseline_lstm.weights.h5
1,msfcal,69026,0.615117,0.657984,0.45,0.493716,0.353223,0.622748,0.450769,models_v6_fixed_abs2/msfcal.weights.h5
